# Module 3: Memory & Persistence

**Day 4 — Agents, LangGraph & MCP**

## What you will learn
- Short-term vs long-term agent memory
- Conversation buffer (the simplest memory)
- `MemorySaver` (InMemorySaver) with thread isolation
- State history and time travel
- `SqliteSaver` for persistent memory

## 1. The Simplest Memory: a Python List

Before any framework, memory is just tracking past messages.

In [ ]:
conversation = []

def chat(user_message: str) -> str:
    conversation.append({"role": "user",      "content": user_message})
    reply = f"You said: '{user_message}'"  # mock LLM
    conversation.append({"role": "assistant", "content": reply})
    return reply

chat("Hello, my name is Arjun")
chat("I work at Zomato in Gurgaon")
chat("Tell me what you know about me")

print(f"{len(conversation)} messages stored:\n")
for msg in conversation:
    print(f"  [{msg['role']:9}] {msg['content']}")

This is exactly what `ConversationBufferMemory` does. The problem: it grows without limit.

**Solutions:**
- `ConversationBufferWindowMemory` — keep last N messages
- `ConversationSummaryMemory` — summarise older turns
- Long-term store — persist to database

## 2. Short-term vs Long-term Memory

| Type | Scope | Cleared when | Use case |
|------|-------|-------------|----------|
| **Short-term** | Current conversation | Session ends | Recent context |
| **Long-term** | Across sessions | Never (manual) | User prefs, history |

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
from day4.memory_persistence import ShortTermMemory, LongTermMemory

stm = ShortTermMemory()
for i in range(5):
    stm.add(f"Message {i+1}")

print("All messages:",    stm.get_recent(10))
print("Last 2 messages:", stm.get_recent(2))

stm.clear()
print("After clear:",     stm.get_recent(10))

ltm = LongTermMemory()
ltm.store("user_name",     "Arjun Sharma")
ltm.store("user_city",     "Gurgaon")
ltm.store("last_project",  "RAG pipeline")
ltm.store("preferred_llm", "GPT-4o")

print("\nName:",   ltm.retrieve("user_name"))
print("Search 'project':", ltm.search("project"))

## 3. LangGraph MemorySaver

When compiled with `MemorySaver`, every `invoke()` saves a checkpoint. The graph remembers previous turns.

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

call_n = [0]
def echo_node(state: ChatState) -> ChatState:
    call_n[0] += 1
    last = state["messages"][-1].content
    return {"messages": [AIMessage(content=f"[{call_n[0]}] Got: {last}")]}

g = StateGraph(ChatState)
g.add_node("echo", echo_node)
g.set_entry_point("echo")
g.add_edge("echo", END)
bot = g.compile(checkpointer=MemorySaver())
cfg = {"configurable": {"thread_id": "demo"}}

for turn in ["Hello", "How are you?", "What is LangGraph?"]:
    result = bot.invoke({"messages": [HumanMessage(turn)]}, config=cfg)

print(f"After 3 turns, total messages: {len(result['messages'])}")
for m in result["messages"]:
    print(f"  {type(m).__name__:<11}: {m.content}")

## 4. Thread Isolation

Each `thread_id` is a completely separate conversation.

In [ ]:
from day4.memory_persistence import build_chatbot_graph, chat_turn, get_conversation_history
from langgraph.checkpoint.memory import MemorySaver

counter = [0]
def mock_llm(messages):
    counter[0] += 1
    return f"Response #{counter[0]}: {messages[-1].content}"

chatbot = build_chatbot_graph(mock_llm, MemorySaver())

chat_turn(chatbot, "Hi, I am Priya from Chennai",  "priya")
chat_turn(chatbot, "Hi, I am Rahul from Mumbai",   "rahul")
chat_turn(chatbot, "What city am I from?",          "priya")

priya = get_conversation_history(chatbot, "priya")
rahul = get_conversation_history(chatbot, "rahul")

print(f"Priya: {len(priya)} messages | Rahul: {len(rahul)} messages")
print(f"Priya last msg: {priya[-1]['content']}")
print(f"Rahul last msg: {rahul[-1]['content']}")

## 5. State History & Time Travel

In [ ]:
from day4.memory_persistence import get_state_history

snapshots = get_state_history(chatbot, "priya")
print(f"Total snapshots: {len(snapshots)}")
for s in snapshots:
    print(f"  step {s['step']:>2}: {s['message_count']} msgs  id={s['checkpoint_id'][:20]}...")

## 6. SqliteSaver — Persist Across Restarts

```python
# pip install langgraph-checkpoint-sqlite aiosqlite
from langgraph.checkpoint.sqlite import SqliteSaver

with SqliteSaver.from_conn_string("conversations.db") as db:
    bot = build_chatbot_graph(my_llm, db)
    # Survives process restarts — history is in the .db file
```